## Importing necessary libraries

In [9]:
import flowermd
import gsd
import gsd.hoomd
import hoomd
import matplotlib.pyplot as plt
import mbuild as mb
import numpy as np
import warnings
from cmeutils.visualize import FresnelGSD
from flowermd.base import Pack, Simulation, System, Molecule
from flowermd.base.forcefield import BaseHOOMDForcefield
from flowermd.library import KremerGrestBeadSpring, LJChain
from flowermd.library.forcefields import BeadSpring
from flowermd.utils import get_target_box_number_density
from mbuild.compound import Compound
from mbuild.lattice import Lattice
import unyt as u
warnings.filterwarnings('ignore')

## Switch to .CPU() if running on CPU, GPU if on GPU

In [10]:
cpu = hoomd.device.CPU()

## Class defining flake geometry

In [11]:
class Graphene(System):
    def __init__(
        self,
        x_repeat,
        y_repeat,
        n_layers,
        base_units=dict(),
        periodicity=(True, True, False),
    ):
        surface = mb.Compound(periodicity=periodicity)
        a = 3**.5
        lattice = Lattice(
            lattice_spacing=[a,a,a],
            lattice_vectors=  [[a,0,0],[a/2,3/2,0],[0,0,1]],
            lattice_points={"A": [[1/3,1/3,0], [2/3, 2/3, 0]]},
        ) 
        Flakium = Compound(name="F", element="F") # defines a carbon atom that will be used to populate lattice points
        layers = lattice.populate(
            compound_dict={"A": Flakium}, x=x_repeat, y=y_repeat, z=n_layers
        ) # populates the lattice using the previously defined carbon atom for every "A" site, repeated in all x,y, and z directions
        surface.add(layers) # adds populated carbon lattice layers to the 'surface' compound, which represents our graphene structure 
        surface.freud_generate_bonds("F", "F", dmin=0.9, dmax=1.1) # generates bonds depending on input distance range, scales with lattice
        surface_mol = Molecule(num_mols=1, compound=surface) # wraps into a Molecule object, creating "1" instance of this molecule

        super(Graphene, self).__init__(
            molecules=[surface_mol],
            base_units=base_units,
        )

    def _build_system(self):
        return self.all_molecules[0]
#OK, so we want to initialize a system with some chains and some flakes

## Forcefield parameters defining WCA interactions, essentially zero attraction, pure repulsion

In [12]:
ff = BeadSpring(
    r_cut=2**(1/6),  
    beads={
        "A": dict(epsilon=1.0, sigma=1.0),  # chains
        "F": dict(epsilon=1.0, sigma=1.0),  # flakes
    },
    bonds={
        "F-F": dict(r0=1.0, k=1000),
        "A-A": dict(r0=1.0, k=1000.0),  # increased k to avoid chain collapse
    },
    angles={
        "A-A-A": dict(t0=2* np.pi / 3., k=100.0),   # moderate stiffness for chains
        "F-F-F": dict(t0=2 * np.pi / 3., k=5000),
    },
    dihedrals={
        "A-A-A-A": dict(phi0=0.0, k=0, d=-1, n=2), #need to turn this on later, messed up with straight chains
        "F-F-F-F": dict(phi0=0.0, k=500, d=-1, n=2),
    }
)

## Simulation parameters

In [43]:
N_chains = 35
initial_dens = 0.001
final_dens = 0.3
N_flakes = 10
chain_length = 10
dt = 0.0005
temp = 3

## Initializing chains, flakes, and system parameters (like initial density -> final density)

In [44]:
kg_chain = LJChain(lengths=chain_length,num_mols=N_chains)
sheet = Graphene(x_repeat=5, y_repeat=5, n_layers=1, periodicity=(False, False, False))
system = Pack(molecules=[Molecule(compound=sheet.all_molecules[0], num_mols=N_flakes), kg_chain], 
              density=initial_dens, packing_expand_factor = 6, seed=2)
target_box = get_target_box_number_density(density=final_dens*u.Unit("nm**-3"),n_beads=(500+(N_chains*10)))

## Output files

In [45]:
gsd = f"{N_chains}_{chain_length}mer{N_flakes}f_{dt}dt.gsd"
log = f"{N_chains}_{chain_length}mer{N_flakes}f_{dt}dt.txt"

## Initializing simulation, shrinking, running, then flushing data into output files.

In [ ]:
sim = Simulation(initial_state=system.hoomd_snapshot, forcefield=ff.hoomd_forces, device=cpu, dt = dt, gsd_write_freq=int(1000), log_file_name = log, gsd_file_name = gsd)
#sim.run_update_volume(final_box_lengths=[(1500/0.8)**1/3, (1500/0.8)**1/3, (1500/0.8)**1/3], kT=6.0, n_steps=5e6,tau_kt=100*sim.dt,period=10,thermalize_particles=True)
sim.run_update_volume(final_box_lengths=target_box, kT=6.0, n_steps=5e6,tau_kt=100*sim.dt,period=10,thermalize_particles=True)
sim.run_NVT(n_steps=1e6, kT=temp, tau_kt=dt*100)
sim.flush_writers()

*Warning*: dihedral.harmonic: specified K <= 0


Initializing simulation state from a gsd.hoomd.Frame.
Step 1000 of 5000000; TPS: 2472.92; ETA: 33.7 minutes
Step 2000 of 5000000; TPS: 3237.2; ETA: 25.7 minutes
Step 3000 of 5000000; TPS: 4014.53; ETA: 20.7 minutes
Step 4000 of 5000000; TPS: 4173.27; ETA: 20.0 minutes
Step 5000 of 5000000; TPS: 4594.32; ETA: 18.1 minutes
Step 6000 of 5000000; TPS: 4928.35; ETA: 16.9 minutes
Step 7000 of 5000000; TPS: 5197.64; ETA: 16.0 minutes
Step 8000 of 5000000; TPS: 5419.79; ETA: 15.4 minutes
Step 9000 of 5000000; TPS: 5603.16; ETA: 14.8 minutes
Step 10000 of 5000000; TPS: 5756.99; ETA: 14.4 minutes
Step 11000 of 5000000; TPS: 5897.58; ETA: 14.1 minutes
Step 12000 of 5000000; TPS: 6017.03; ETA: 13.8 minutes
Step 13000 of 5000000; TPS: 6121.07; ETA: 13.6 minutes
Step 14000 of 5000000; TPS: 6213.08; ETA: 13.4 minutes
Step 15000 of 5000000; TPS: 6294.35; ETA: 13.2 minutes
Step 16000 of 5000000; TPS: 6367.51; ETA: 13.0 minutes
Step 17000 of 5000000; TPS: 6432.6; ETA: 12.9 minutes
Step 18000 of 5000000;

## Visualization of final frame. Can adjust to whatever frame if you'd like

In [ ]:
sim_visualizer = FresnelGSD(gsd_file=gsd, frame=-1, view_axis=(1, 1, 1))
sim_visualizer.view()